# Phase 2 - SQL Data Cleaning   
**Dataset:** Bank Customer Churn Modelling   
**Goal:** Drop irrelevant columns, standardise types, and engineer churn features   
**Input:** `Data/churn.db` - `churn_raw`   
**Output:** `data/churn.db` - `churn_clean`

### Audit the raw data

In [3]:
# imports
import pandas as pd
import sqlite3


conn = sqlite3.connect('../data/churn.db')
df =pd.read_sql("SELECT * FROM churn_raw", conn)

#column audit
audit = pd.DataFrame({
    'dtype': df.dtypes,
    'nulls': df.isnull().sum(),
    'unique_vals': df.nunique(),
    'sample_val': df.iloc[0]
})

print(audit)
conn.close()

                   dtype  nulls  unique_vals sample_val
RowNumber          int64      0        10000          1
CustomerId         int64      0        10000   15634602
Surname              str      0         2932   Hargrave
CreditScore        int64      0          460        619
Geography            str      0            3     France
Gender               str      0            2     Female
Age                int64      0           70         42
Tenure             int64      0           11          2
Balance          float64      0         6382        0.0
NumOfProducts      int64      0            4          1
HasCrCard          int64      0            2          1
IsActiveMember     int64      0            2          1
EstimatedSalary  float64      0         9999  101348.88
Exited             int64      0            2          1


### Drop irrelevant columns & cast types

In [5]:
conn = sqlite3.connect('../data/churn.db')

df = pd.read_sql("SELECT * FROM churn_raw", conn)

#Drop columns with no analytical value
cols_to_drop = ['RowNumber', 'Surname']
df_trimmed = df.drop(columns = cols_to_drop)

#confirm
print(f"Columns before: {df.shape[1]} - after: {df_trimmed.shape[1]}")
print(df_trimmed.columns.tolist())

conn.close()

Columns before: 14 - after: 12
['CustomerId', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited']


### Engineer Features

In [7]:
conn = sqlite3.connect('../data/churn.db')
df = pd.read_sql("SELECT * FROM churn_raw", conn)
df = df.drop(columns=['RowNumber', 'Surname'])

# feature 1 - Age band
def age_band(age):
    if age < 30: return 'Under 30'
    elif age < 40: return '30 - 39'
    elif age < 50: return '40 - 49'
    elif age < 60: return '50 - 59'
    else: return '60+'

df['age_band'] = df['Age'].apply(age_band)

#Feature 2 - Balance Segment
def balance_segment(bal):
    if bal == 0: return 'Zero Balance'
    elif bal < 50000: return 'Low (< 50k)'
    elif bal < 100000: return 'Mid (50k - 100k)'
    elif bal < 150000: return 'High (100k - 150k)'
    else: return 'Very High (150k+)'

df['balance_segment'] = df['Balance'].apply(balance_segment)

# Feature 3 - Credit score band
def credit_band(score):
    if score < 580: return 'Poor'
    elif score < 670: return 'Fair'
    elif score < 740: return 'Good'
    elif score < 800: return 'Very Good'
    else: return 'Exceptional'

df['credit_band'] = df['CreditScore'].apply(credit_band)

# Feature 4 - Customer value tier
# Combines products held, active membership and balance into a single score
 
def value_tir(row):
    score = 0
    if row['Balance'] >= 100000: score += 3
    elif row['Balance'] >= 50000: score += 1
    if row['NumOfProducts'] >= 3: score += 2
    elif row['NumOfProducts'] == 2: score += 1
    if row['IsActiveMember'] == 1: score += 2
    if row['Tenure'] >= 5: score += 1

    if score >= 5: return 'High Value'
    elif score >= 2: return 'Mid Value'
    else: return 'Low Value'

df['value_tier'] = df.apply(value_tir, axis=1)

# verify all four features
print(df['age_band'].value_counts())
print()
print(df['balance_segment'].value_counts())
print()
print(df['credit_band'].value_counts())
print()
print(df['value_tier'].value_counts())

age_band
30 - 39     4346
40 - 49     2618
Under 30    1641
50 - 59      869
60+          526
Name: count, dtype: int64

balance_segment
High (100k - 150k)    3830
Zero Balance          3617
Mid (50k - 100k)      1509
Very High (150k+)      969
Low (< 50k)             75
Name: count, dtype: int64

credit_band
Fair           3331
Good           2428
Poor           2362
Very Good      1224
Exceptional     655
Name: count, dtype: int64

value_tier
Mid Value     5677
High Value    3114
Low Value     1209
Name: count, dtype: int64


### Build the clean table (churn_clean)

In [8]:
conn = sqlite3.connect('../data/churn.db')
cur = conn.cursor()

#Drop if exists to rerun cleanly
cur.execute("DROP TABLE IF EXISTS churn_clean")

df.to_sql('churn_clean', conn, if_exists='replace', index=False)

print(f"churn_clean created: {df.shape[0]:,} rows * {df.shape[1]} columns")
conn.close()

churn_clean created: 10,000 rows * 16 columns


### Key Summary queries

In [9]:
conn = sqlite3.connect('../data/churn.db')

# Churn by geography
geo = pd.read_sql("""
    SELECT
        Geography,
        COUNT(*) As total_customers,
        SUM(Exited) As churned,
        ROUND(AVG(CAST(Exited AS FLOAT)) * 100,2) As churn_rate_pct
    FROM churn_clean
    GROUP BY Geography
    ORDER BY churn_rate_pct DESC
""", conn)
print("=== Churn by Geography ===")
print(geo)

=== Churn by Geography ===
  Geography  total_customers  churned  churn_rate_pct
0   Germany             2509      814           32.44
1     Spain             2477      413           16.67
2    France             5014      810           16.15


In [10]:
# Churn by age band
age = pd.read_sql(""" 
    SELECT
        age band,
        COUNT(*) AS total_customers,
        SUM(Exited) AS churned,
        ROUND(AVG(CAST(Exited AS FLOAT)) * 100,2) AS churn_rate_pct
    FROM churn_clean
    GROUP BY age_band
    ORDER BY churn_rate_pct DESC

""", conn)

print("\n=== Churn by Age Band ===")
print(age)


=== Churn by Age Band ===
   band  total_customers  churned  churn_rate_pct
0    50              869      487           56.04
1    42             2618      806           30.79
2    61              526      147           27.95
3    39             4346      473           10.88
4    29             1641      124            7.56


In [11]:
# Churn by value tier
value = pd.read_sql("""
    SELECT
        value_tier,
        COUNT(*) AS total_customers,
        SUM(Exited) AS churned,
        ROUND(AVG(CAST(Exited AS FLOAT)) * 100,2) AS churn_rate_pct
    FROM churn_clean
    GROUP BY value_tier
    ORDER BY churn_rate_pct DESC
""", conn)
print("\n=== Churn by Value Tier ===")
print(value)


conn.close()


=== Churn by Value Tier ===
   value_tier  total_customers  churned  churn_rate_pct
0   Low Value             1209      323           26.72
1  High Value             3114      647           20.78
2   Mid Value             5677     1067           18.80


### Quick Checks

In [12]:
conn = sqlite3.connect('../data/churn.db')

checks = {
    "No NULL CustomerIds":     "SELECT COUNT(*) FROM churn_clean WHERE CustomerId IS NULL",
    "No duplicate customers":  "SELECT COUNT(*) FROM (SELECT CustomerId FROM churn_clean GROUP BY CustomerId HAVING COUNT(*) > 1)",
    "Valid Exited values":     "SELECT COUNT(*) FROM churn_clean WHERE Exited NOT IN (0, 1)",
    "Valid Gender values":     "SELECT COUNT(*) FROM churn_clean WHERE Gender NOT IN ('Male', 'Female')",
    "Valid Geography values":  "SELECT COUNT(*) FROM churn_clean WHERE Geography NOT IN ('France', 'Germany', 'Spain')",
    "No NULL age_band":        "SELECT COUNT(*) FROM churn_clean WHERE age_band IS NULL",
    "No NULL value_tier":      "SELECT COUNT(*) FROM churn_clean WHERE value_tier IS NULL",
}

print("=== Data Quality Report ===\n")
all_passed = True
for check_name, query in checks.items():
    result = pd.read_sql(query, conn).iloc[0, 0]
    status = " PASS" if result == 0 else f" FAIL ({result} rows)"
    if result != 0:
        all_passed = False
    print(f"  {status}  —  {check_name}")

print(f"\n{'All checks passed! Ready for EDA.' if all_passed else 'Review failing checks above.'}")
conn.close()

=== Data Quality Report ===

   PASS  —  No NULL CustomerIds
   PASS  —  No duplicate customers
   PASS  —  Valid Exited values
   PASS  —  Valid Gender values
   PASS  —  Valid Geography values
   PASS  —  No NULL age_band
   PASS  —  No NULL value_tier

All checks passed! Ready for EDA.


### Export churn_clean to CSV

In [13]:
conn = sqlite3.connect('../data/churn.db')
df_clean = pd.read_sql("SELECT * FROM churn_clean", conn)
df_clean.to_csv('../data/churn_clean.csv', index=False)
print(f"Exported {len(df_clean):,} rows to data/churn_clean.csv")
conn.close()

Exported 10,000 rows to data/churn_clean.csv


## END OF PHASE 2